# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)
# Access and pretty-print core metadata
meta = dataset.metadata
meta_json = meta.to_json()
print('Dataset name:', meta_json.get('name'))
print('Description:', meta_json.get('description'))
print('Version:', meta_json.get('version'))
print('License:', meta_json.get('license'))

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we enumerate the available record sets (tables) in this Croissant package and their associated fields, referencing them by their `@id`.

In [ ]:
# List all record sets and their fields by @id
record_sets = meta.record_sets
print(f'Found {len(record_sets)} record set(s):')
overview = {}
for rs in record_sets:
    rs_id = rs.id  # @id of the record set
    print(f"- RecordSet @id: {rs_id} | Name: '{rs.name}'")
    fields = rs.fields
    overview[rs_id] = [f.id for f in fields]
    for field in fields:
        print(f"    - Field name: {field.name} | Field @id: {field.id} | DataType: {field.data_type}")

if len(record_sets) == 0:
    print("No record sets found in metadata. Attempting to list data distributions.")
    # If not directly present, try listing primary data distributions
    for dist in meta.distribution:
        print(f"Distribution: {dist.id if hasattr(dist, 'id') else dist}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Choose the appropriate record set `@id` from the list above. For this dataset, we will select the primary table (if available).

In [ ]:
# If the Croissant metadata exposes record_sets, select the main record set by @id.
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

# If no record sets are provided in the metadata, you may need to refer to `distribution` for files/extractions.
if record_set_ids:
    for record_set_id in record_set_ids:
        print(f"Loading records from RecordSet @id: {record_set_id}")
        try:
            records = list(dataset.records(record_set=record_set_id))
            if records:
                df = pd.DataFrame(records)
                dataframes[record_set_id] = df
                print(f"Loaded dataframe for {record_set_id} with shape {df.shape}")
                print('Columns:', list(df.columns))
            else:
                print(f"No records found for RecordSet @id: {record_set_id}")
        except Exception as e:
            print(f"Error loading records for {record_set_id}: {e}")
else:
    print("No record sets detected, cannot extract tables by @id.")
    # As fallback, try direct extraction from distribution if possible

# For demonstration, select the first available dataframe
if dataframes:
    main_record_set_id = next(iter(dataframes.keys()))
    print(f"Example RecordSet @id: {main_record_set_id}")
    print(dataframes[main_record_set_id].head())
else:
    main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes examples such as removing outliers, examining distributions, or summarizing key columns. All references use the field `@id`s.

In [ ]:
# Ensure a dataframe is available
if main_record_set_id and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]

    # Try to auto-select a numeric column by inspecting dtypes
    numeric_cols = df.select_dtypes(include=['number', 'float', 'int']).columns.tolist()
    if not numeric_cols:
        # Attempt conversion of likely numeric columns
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='ignore')
            except Exception:
                continue
        numeric_cols = df.select_dtypes(include=['number', 'float', 'int']).columns.tolist()

    if numeric_cols:
        numeric_field = numeric_cols[0]  # Use the first available numeric field by @id
        print(f"Using numeric field for EDA: {numeric_field}")
        # Example filtering: values > 10, if that's meaningful (otherwise adjust threshold)
        threshold = 10
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered {len(filtered_df)} rows with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalize the numeric field (z-score)
        filtered_df[numeric_field + '_normalized'] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
            filtered_df[numeric_field].std()
        )
        print(f"\nNormalized '{numeric_field}' for filtered records (first 5 rows):")
        print(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

        # Attempt to group by a categorical/text field
        group_candidates = df.select_dtypes(include=['object', 'category']).columns.tolist()
        group_field = group_candidates[0] if group_candidates else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nMean '{numeric_field}' grouped by '{group_field}':")
            print(grouped_df.head())
        else:
            print('No categorical field available for grouping.')
    else:
        print('No numeric fields found in the record set for EDA.')
else:
    print('No suitable dataframe available for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.
Below is a sample visualization showing the distribution of the selected numeric field and, if possible, the grouped means by category.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and main_record_set_id in dataframes and 'numeric_field' in locals() and numeric_field in df.columns:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    if 'group_field' in locals() and group_field and group_field in df.columns:
        plt.figure(figsize=(10,5))
        # Average value by group (top 20, for clarity)
        group_means = df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)[:20]
        sns.barplot(x=group_means.index, y=group_means.values)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.ylabel(f"Mean {numeric_field}")
        plt.xlabel(f"{group_field}")
        plt.xticks(rotation=90)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Loaded Croissant schema-based dataset metadata and records.
- Inspected record sets, fields, and field `@id`s as unique references.
- Demonstrated EDA using numeric and grouping fields for filtering and normalization.
- Visualized data distributions and group summarization.

This exploration can serve as a starting point for further analysis, machine learning, or domain-specific investigations into protein abundance and characteristics in extracellular vesicles from human mast cells.